In [2]:
import os
import sys
import json
import time
import kaggle
from kagglehub.competition import competition_download

import numpy as np
import pandas as pd


from sklearn.model_selection import StratifiedKFold, train_test_split
import xgboost as xgb
from sklearn.metrics import roc_auc_score, classification_report
import optuna


os.environ['KAGGLE_USERNAME'] = json.load(open('/home/osman-tekdamar/.config/kaggle/kaggle.json'))['username']
os.environ['KAGGLE_KEY'] = json.load(open('/home/osman-tekdamar/.config/kaggle/kaggle.json'))['key']
path = competition_download('ing-hubs-turkiye-datathon')

In [3]:
customer_history = pd.read_csv(f"{path}/customer_history.csv")
customers = pd.read_csv(f"{path}/customers.csv")
referance_data = pd.read_csv(f"{path}/referance_data.csv")
referance_data_test = pd.read_csv(f"{path}/referance_data_test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv") 

In [ ]:
from metrics import recall_at_k, lift_at_k, convert_auc_to_gini, ing_hubs_datathon_metric

In [5]:
train_data = customers.merge(referance_data, "right", on="cust_id")
test_data = customers.merge(referance_data_test, "right", on="cust_id")

In [6]:
train_data = train_data.drop(["cust_id", "ref_date"], axis=1)
test_data = test_data.drop(["cust_id", "ref_date"], axis=1)

In [7]:
train_data

,gender,age,province,religion,work_type,work_sector,tenure,churn
0,F,64,NOH,U,Part-time,Technology,135,0
1,F,22,ZUI,C,Student,NaN,47,0
2,M,27,ZUI,U,Full-time,Finance,108,1
3,F,40,NOH,U,Unemployed,NaN,187,1
4,F,64,GEL,U,Part-time,Public Sector,218,0
...,...,...,...,...,...,...,...,...
133282,F,54,GEL,C,Part-time,Public Sector,217,0
133283,M,47,GEL,C,Full-time,Public Sector,37,0
133284,F,66,NOB,C,Retired,NaN,227,0
133285,F,31,ZUI,U,Self-employed,Education,156,1


In [8]:
num_cols = [col for col in test_data.columns if test_data[col].dtype != object]
cat_cols = [col for col in test_data.columns if test_data[col].dtype == object]

In [9]:
train_cat_df = pd.get_dummies(train_data[cat_cols], drop_first=True)
test_cat_df = pd.get_dummies(test_data[cat_cols], drop_first=True)

In [10]:
train_num_df = pd.DataFrame(train_data[num_cols].values, columns=num_cols)
test_num_df = pd.DataFrame(test_data[num_cols].values, columns=num_cols)

In [11]:
new_train = pd.concat([train_cat_df, train_num_df,train_data["churn"]], axis=1)
new_test  =pd.concat([test_cat_df, test_num_df], axis=1)

In [12]:
X = new_train.drop("churn", axis=1)
y = new_train["churn"]

In [13]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)

In [14]:
def objective(trial, X, y):
    """
    Optuna'nın her bir denemede çalıştıracağı ve özel metriği maksimize edeceği fonksiyon.
    """
    
    # Hiperparametre Arama Uzayını Tanımla
    param = {
        'objective': 'binary:logistic',
        'eval_metric': 'logloss',
        'booster': 'gbtree',
        'device': 'gpu',
        'early_stopping_rounds': 50,
        'n_estimators': 1000,
        'lambda': trial.suggest_float('lambda', 1e-8, 1.0, log=True),
        'alpha': trial.suggest_float('alpha', 1e-8, 1.0, log=True),
        'max_depth': trial.suggest_int('max_depth', 3, 9),
        'eta': trial.suggest_float('eta', 0.01, 0.3, log=True),
        'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
    }

    # Dengesiz veri için kritik olan sınıf ağırlığını hesapla
    scale_pos_weight = np.sum(y == 0) / np.sum(y == 1)
    param['scale_pos_weight'] = scale_pos_weight

    # StratifiedKFold ile Çapraz Doğrulama
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    
    for train_idx, val_idx in cv.split(X, y):
        X_train_fold, y_train_fold = X.iloc[train_idx], y.iloc[train_idx]
        X_val_fold, y_val_fold = X.iloc[val_idx], y.iloc[val_idx]
        
        model = xgb.XGBClassifier(**param, random_state=42)
        
        # Modeli eğit (Early stopping ile aşırı öğrenmeyi engelle)
        model.fit(X_train_fold, y_train_fold,
                  eval_set=[(X_val_fold, y_val_fold)],
                  verbose=False)
        
        preds_proba = model.predict_proba(X_val_fold)[:, 1]
        
        # Özel değerlendirme metriğini kullanarak skoru hesapla
        custom_score = ing_hubs_datathon_metric(y_val_fold, preds_proba)
        scores.append(custom_score)

    # Ortalamayı döndür. Optuna bu değeri maksimize etmeye çalışacak.
    return np.mean(scores)


In [15]:

# =============================================================================
# 5. Optimizasyon Sürecini Başlatma
# =============================================================================
print("--- Optuna Optimizasyonu Başlatılıyor ---")
# 'direction="maximize"' ile özel metriğimizin en yüksek değerini arıyoruz
study = optuna.create_study(direction='maximize')

# Optimizasyonu n_trials kadar deneme ile çalıştır
study.optimize(lambda trial: objective(trial, X_train, y_train), n_trials=50, show_progress_bar=True)

print("Optimizasyon tamamlandı.\n")
print("--- En İyi Optimizasyon Sonuçları ---")
best_trial = study.best_trial
print(f"En İyi Değer (Ortalama Özel Metrik): {best_trial.value:.4f}")
print("En İyi Parametreler:")
for key, value in best_trial.params.items():
    print(f"  {key}: {value}")
print("-" * 30, "\n")



[I 2026-07-16 14:26:06,019] A new study created in memory with name: no-name-830c5089-55ed-4fb6-899d-16d1b12a63f5


--- Optuna Optimizasyonu Başlatılıyor ---


  0%|          | 0/50 [00:00<?, ?it/s]/home/osman-tekdamar/Projects/forCV/ING_Datathon/.claude/worktrees/init-claude-md/.venv/lib/python3.12/site-packages/xgboost/core.py:729: UserWarning: [14:26:08] WARNING: /workspace/src/common/error_msg.cc:58: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)
Best trial: 0. Best value: 0.36996:   2%|▏         | 1/50 [00:08<07:07,  8.72s/it]

[I 2026-07-16 14:26:14,734] Trial 0 finished with value: 0.3699598769104014 and parameters: {'lambda': 0.017707446803856726, 'alpha': 0.05369251377322737, 'max_depth': 5, 'eta': 0.028467227337943702, 'gamma': 0.0004920776476630352, 'colsample_bytree': 0.6882812279230995, 'subsample': 0.7191226841695918, 'min_child_weight': 2}. Best is trial 0 with value: 0.3699598769104014.


Best trial: 0. Best value: 0.36996:   4%|▍         | 2/50 [00:14<05:45,  7.19s/it]

...

[I 2026-07-16 14:30:14,634] Trial 47 finished with value: 0.402801659904293 and parameters: {'lambda': 0.23392220740028152, 'alpha': 0.01858749849684231, 'max_depth': 4, 'eta': 0.031848345322226176, 'gamma': 1.6470196040967024e-07, 'colsample_bytree': 0.8095174803219982, 'subsample': 0.5459586530590931, 'min_child_weight': 5}. Best is trial 25 with value: 0.42334938113341297.


Best trial: 25. Best value: 0.423349:  98%|█████████▊| 49/50 [04:10<00:03,  3.35s/it]

[I 2026-07-16 14:30:16,378] Trial 48 finished with value: 0.4193457422128876 and parameters: {'lambda': 0.03080128374475263, 'alpha': 0.5302707046560867, 'max_depth': 3, 'eta': 0.020633137316002186, 'gamma': 6.660862167083589e-06, 'colsample_bytree': 0.78063949873716, 'subsample': 0.5927678535515718, 'min_child_weight': 6}. Best is trial 25 with value: 0.42334938113341297.


Best trial: 25. Best value: 0.423349: 100%|██████████| 50/50 [04:17<00:00,  5.14s/it]

[I 2026-07-16 14:30:23,210] Trial 49 finished with value: 0.3960715143484939 and parameters: {'lambda': 0.03828223494377015, 'alpha': 0.9949839062003648, 'max_depth': 3, 'eta': 0.019587640336611905, 'gamma': 6.053100881744492e-06, 'colsample_bytree': 0.7733304982342011, 'subsample': 0.9917622284643912, 'min_child_weight': 6}. Best is trial 25 with value: 0.42334938113341297.
Optimizasyon tamamlandı.

--- En İyi Optimizasyon Sonuçları ---
En İyi Değer (Ortalama Özel Metrik): 0.4233
En İyi Parametreler:
  lambda: 0.0034506418736171343
  alpha: 0.915831003421079
  max_depth: 3
  eta: 0.02005300888792457
  gamma: 0.04043083922166823
  colsample_bytree: 0.8082353691901047
  subsample: 0.5372520595189859
  min_child_weight: 6
------------------------------ 



In [16]:

# =============================================================================
# 6. Final Modelin Eğitilmesi ve Değerlendirilmesi
# =============================================================================
print("--- Final Model Eğitiliyor ve Değerlendiriliyor ---")
# Optuna'nın bulduğu en iyi parametreleri al
best_params = best_trial.params

# Optimizasyon dışında kalan sabit parametreleri ekle
best_params['scale_pos_weight'] = np.sum(y_train == 0) / np.sum(y_train == 1)
best_params['n_estimators'] = 2000  # Early stopping için yüksek bir değer
best_params['random_state'] = 42
best_params['objective'] = 'binary:logistic'
best_params["early_stopping_rounds"] = 50

# Final modeli en iyi parametrelerle oluştur
final_model = xgb.XGBClassifier(**best_params)

# Final modelin eğitiminde de early stopping kullanmak iyi bir pratiktir.
# Bunun için eğitim verisinden küçük bir validasyon seti ayırabiliriz.
X_train_part, X_val_part, y_train_part, y_val_part = train_test_split(
    X_train, y_train, test_size=0.1, random_state=42, stratify=y_train
)

final_model.fit(X_train_part, y_train_part,
                eval_set=[(X_val_part, y_val_part)],
                verbose=False)

# Daha önce hiç görülmemiş TEST VERİSİ üzerinde tahmin yap
y_pred_proba_test = final_model.predict_proba(X_test)[:, 1]

# Test seti üzerinde özel metrik skorunu hesapla
final_custom_score = ing_hubs_datathon_metric(y_test, y_pred_proba_test)
print(f"Test Seti Üzerindeki Özel Metrik Skoru: {final_custom_score:.4f}\n")

# Özel metriği oluşturan alt metriklerin dökümünü de alalım
test_auc = roc_auc_score(y_test, y_pred_proba_test)
test_gini = convert_auc_to_gini(test_auc)
test_recall10 = recall_at_k(y_test, y_pred_proba_test, k=0.1)
test_lift10 = lift_at_k(y_test, y_pred_proba_test, k=0.1)

print("--- Test Seti Detaylı Metrikler ---")
print(f"Gini: {test_gini:.4f}")
print(f"Recall@10%: {test_recall10:.4f}")
print(f"Lift@10%: {test_lift10:.4f}\n")

# Sınıflandırma raporu için bir eşik değeri belirleyelim (örn: 0.5)
y_pred_class_test = (y_pred_proba_test > 0.5).astype(int)
print("--- Test Seti Classification Report (0.5 Eşik Değeri ile) ---")
print(classification_report(y_test, y_pred_class_test))
print("=" * 70)

--- Final Model Eğitiliyor ve Değerlendiriliyor ---
Test Seti Üzerindeki Özel Metrik Skoru: 0.4156

--- Test Seti Detaylı Metrikler ---
Gini: 0.0442
Recall@10%: 0.1138
Lift@10%: 1.1383

--- Test Seti Classification Report (0.5 Eşik Değeri ile) ---
              precision    recall  f1-score   support

           0       0.86      0.55      0.67     28604
           1       0.15      0.48      0.23      4718

    accuracy                           0.54     33322
   macro avg       0.51      0.51      0.45     33322
weighted avg       0.76      0.54      0.61     33322



In [17]:
best_params.pop("early_stopping_rounds")

50

In [18]:
final_model = xgb.XGBClassifier(**best_params)
final_model.fit(X,y)

,objective,'binary:logistic'
,base_score,None
,booster,None
,callbacks,None
,colsample_bylevel,None
,colsample_bynode,None
,colsample_bytree,0.8082353691901047
,device,None
,early_stopping_rounds,None
,enable_categorical,False
,eval_metric,None


In [19]:
sample_submission["churn"] = final_model.predict_proba(new_test)[:, 1]

In [20]:
sample_submission.to_csv('/tmp/submission.csv', index=False)
kaggle.api.competition_submit(
    file_name='/tmp/submission.csv', 
    message='xgb with Optuna kfold', 
    competition='ing-hubs-turkiye-datathon'
)

100%|██████████| 713k/713k [00:01<00:00, 564kB/s] 


{"message": "Successfully submitted to ING Hubs T\u00fcrkiye Datathon", "ref": 54759441}